In [2]:
import torch
import torch.nn as nn
import torch.optim as optim
import pandas as pd
import numpy as np
from torch.utils.data import TensorDataset, DataLoader

print("PyTorch version:", torch.__version__)

PyTorch version: 2.10.0+cpu


In [3]:
import pandas as pd

file_name = 'RS126.data.txt' # <-- CHANGE THIS to your file's name!
window_size = 13
pad_length = window_size // 2

X_data = [] # The 13-letter windows
Y_data = [] # The 1-letter target shapes

print("Reading biological data...")

# 1. Open and read the file
with open(file_name, 'r') as file:
    lines = file.readlines() 

# 2. Loop through the lines 2 at a time (Sequence, then Structure)
# Let's process the first 10 proteins (20 lines) 
for i in range(0, 20, 2): 
    
    seq = lines[i].strip()       
    struct = lines[i+1].strip()  
    
    # 3. Pad the sequence with 'X'
    padded_seq = ("X" * pad_length) + seq + ("X" * pad_length)
    
    # 4. Slide the window!
    for j in range(len(seq)):
        window = padded_seq[j : j + window_size]
        target = struct[j]
        
        X_data.append(window)
        Y_data.append(target)

print(f"Successfully chopped data into {len(X_data)} training examples!\n")

# 5. Convert it into a Pandas DataFrame
df = pd.DataFrame({
    'Window_Input': X_data,
    'Target_Structure': Y_data
})

# 6. Print the first 15 rows to the VS Code Terminal
print("--- First 15 Windows ---")
print(df.head(15))

Reading biological data...
Successfully chopped data into 1234 training examples!

--- First 15 Windows ---
     Window_Input Target_Structure
0   XXXXXXAPAFSVS                C
1   XXXXXAPAFSVSP                C
2   XXXXAPAFSVSPA                E
3   XXXAPAFSVSPAS                E
4   XXAPAFSVSPASG                E
5   XAPAFSVSPASGA                E
6   APAFSVSPASGAS                E
7   PAFSVSPASGASD                C
8   AFSVSPASGASDG                C
9   FSVSPASGASDGQ                C
10  SVSPASGASDGQS                C
11  VSPASGASDGQSV                C
12  SPASGASDGQSVS                C
13  PASGASDGQSVSV                C
14  ASGASDGQSVSVS                C


In [4]:
# 1. CONVERT Y STRINGS (C, E, H) TO INTEGERS (0, 1, 2)
shape_mapping = {'C': 0, 'E': 1, 'H': 2}
Y_ints = [shape_mapping[shape] for shape in Y_data]
Y_tensor = torch.tensor(Y_ints, dtype=torch.long) # PyTorch expects LongTensor for class labels

# 2. ONE-HOT ENCODE X WINDOWS (Strings to Numbers)
# Alphabet contains 20 standard amino acids + 'X' (padding)
alphabet = "ACDEFGHIKLMNPQRSTVWYX"
char_to_idx = {char: i for i, char in enumerate(alphabet)}
window_size = 13
vocab_size = len(alphabet) # 21

# Initialize a giant matrix of zeros: [Total_Rows, 13 * 21]
X_onehot = torch.zeros(len(X_data), window_size * vocab_size)

for row_idx, window in enumerate(X_data):
    for char_idx, char in enumerate(window):
        if char in char_to_idx:
            # Calculate the exact column index for this character in this position
            col_idx = (char_idx * vocab_size) + char_to_idx[char]
            X_onehot[row_idx, col_idx] = 1.0

print(f"Finished One-Hot Encoding!")
print(f"X_onehot shape (Inputs): {X_onehot.shape}") # Should be [Total_Rows, 273]
print(f"Y_tensor shape (Targets): {Y_tensor.shape}") # Should be [Total_Rows]
print(len(X_data)) # Should be the same number of rows

Finished One-Hot Encoding!
X_onehot shape (Inputs): torch.Size([1234, 273])
Y_tensor shape (Targets): torch.Size([1234])
1234


In [34]:
X_onehot.shape


torch.Size([1234, 273])

In [5]:
# Combine X and Y into a single PyTorch Dataset object
dataset = TensorDataset(X_onehot, Y_tensor)

# Create a DataLoader to automatically shuffle data and slice it into mini-batches
BATCH_SIZE = 32  # Process 32 window-rows at a time
train_loader = DataLoader(dataset, batch_size=BATCH_SIZE, shuffle=True)

print(f"Dataset successfully loaded into DataLoader.")
print(f"Total mini-batches per epoch: {len(train_loader)}")

Dataset successfully loaded into DataLoader.
Total mini-batches per epoch: 39


In [6]:
class MiniFoldFFNN(nn.Module):
    def __init__(self, input_size, hidden_size, output_size):
        super(MiniFoldFFNN, self).__init__()
        
        # Layer 1: Takes 273 inputs, maps to hidden_size (e.g., 64 neurons)
        # PyTorch AUTOMATICALLY initializes random weights and biases here!
        self.fc1 = nn.Linear(input_size, hidden_size)
        
        # Activation Function (adds non-linearity)
        self.relu = nn.ReLU()
        
        # Layer 2: Maps hidden_size to 3 outputs (Classes: C, E, H)
        self.fc2 = nn.Linear(hidden_size, output_size)
        
    def forward(self, x):
        # This is the "Feed-Forward" pass
        out = self.fc1(x)     # Multiply inputs by random weights + add bias
        out = self.relu(out)  # Pass through ReLU activation
        out = self.fc2(out)   # Final layer to get 3 raw scores (logits) for C, E, H
        return out

In [7]:
train_loader.batch_size
train_loader

In [16]:
# Dimensions
INPUT_SIZE = 13 * 21  # 273
HIDDEN_SIZE = 256      # 256 neurons in the hidden layer
OUTPUT_SIZE = 3       # 3 possible shapes (C, E, H)

# Create the model
model = MiniFoldFFNN(INPUT_SIZE, HIDDEN_SIZE, OUTPUT_SIZE)

# 1. LOSS FUNCTION: Categorical Cross-Entropy (Perfect for discrete classes C, E, H)
criterion = nn.CrossEntropyLoss()

# 2. OPTIMIZER: Adam (This is the algorithm that handles Backpropagation and updates weights)
optimizer = torch.optim.Adam(model.parameters(), lr=0.001)

In [17]:
# model
firstBatch = next(iter(train_loader))
firstBatch

[tensor([[1., 0., 0.,  ..., 0., 0., 0.],
         [0., 0., 0.,  ..., 0., 0., 0.],
         [0., 1., 0.,  ..., 0., 0., 0.],
         ...,
         [0., 0., 0.,  ..., 0., 0., 0.],
         [0., 1., 0.,  ..., 0., 0., 0.],
         [0., 0., 0.,  ..., 0., 0., 0.]]),
 tensor([1, 0, 0, 1, 1, 0, 1, 0, 0, 0, 1, 0, 0, 0, 0, 1, 0, 1, 0, 0, 1, 0, 0, 0,
         0, 0, 1, 2, 1, 0, 0, 0])]

In [27]:
STEPS = 100  # Number of times the AI reads the entire dataset

print("--- STARTING TRAINING ---")
for epoch in range(STEPS):
    running_loss = 0.0
    
    # Loop through the data in mini-batches of 64!
    for batch_X, batch_Y in train_loader:
        
        # 1. FORWARD PASS: Pass mini-batch through the network
        outputs = model(batch_X)
        # print(f"batch_X Shape: {batch_X.shape}")
        # print(f"batch_Y Shape: {batch_Y.shape}")
        #print(f"Batch Outputs Shape: {outputs.shape}") 

        # 2. CALCULATE LOSS: Compare AI's guesses to real answers
        loss = criterion(outputs, batch_Y)
        
        # 3. BACKPROPAGATION:
        optimizer.zero_grad()  # Clear old gradients
        loss.backward()        # Calculate calculus gradients (Backprop!)
        optimizer.step()       # Update weights & biases
        
        running_loss += loss.item()
        
    # Calculate average loss for this epoch and print it
    avg_loss = running_loss / len(train_loader)
    if(epoch % 10 == 0):  # Print every 100 epochs
        print(f"[{epoch}/{STEPS}] | Average Loss: {avg_loss}")

print("--- TRAINING COMPLETE ---")

--- STARTING TRAINING ---
[0/100] | Average Loss: 9.552025836967459e-11
[10/100] | Average Loss: 0.0
[20/100] | Average Loss: 1.9104051673934919e-10
[30/100] | Average Loss: 9.552025836967459e-11
[40/100] | Average Loss: 4.77601291848373e-10
[50/100] | Average Loss: 5.731215502180476e-10
[60/100] | Average Loss: 1.3372836057885415e-09
[70/100] | Average Loss: 8.596823253270713e-10
[80/100] | Average Loss: 3.3219823017761065e-09
[90/100] | Average Loss: 2.388006442161511e-09
--- TRAINING COMPLETE ---


In [28]:
def predict_protein_structure(protein_seq, trained_model, window_size=13, alphabet="ACDEFGHIKLMNPQRSTVWYX"):
    pad_length = window_size // 2
    padded_seq = ("X" * pad_length) + protein_seq + ("X" * pad_length)
    
    char_to_idx = {char: i for i, char in enumerate(alphabet)}
    vocab_size = len(alphabet)
    
    # 2. Build one-hot encoded windows for every single letter in this protein
    X_inference = torch.zeros(len(protein_seq), window_size * vocab_size)
    
    for i in range(len(protein_seq)):
        window = padded_seq[i : i + window_size]
        for char_idx, char in enumerate(window):
            if char in char_to_idx:
                col_idx = (char_idx * vocab_size) + char_to_idx[char]
                X_inference[i, col_idx] = 1.0
                
    # 3. Pass through the model (torch.no_grad() speeds things up since we aren't training)
    with torch.no_grad():
        outputs = trained_model(X_inference) # Shape: [length_of_protein, 3]
        predicted_indices = torch.argmax(outputs, dim=1)
        int_to_shape = {0: 'C', 1: 'E', 2: 'H'}
    predicted_chars = [int_to_shape[int(idx.item())] for idx in predicted_indices]
    
    # Join the list of characters back into a single string
    return "".join(predicted_chars)

# --- RUN THE TEST ---
test_input = "FVNQHLCGSHLVEALYLVCGERGFFYTPKA"
expected_output = "CCCCCCCCHHHHHHHHHHHHHHCECCCCCC"

model_prediction = predict_protein_structure(test_input, model)

print(f"Input:    {test_input}")
print(f"Expected: {expected_output}")
print(f"Predicted:{model_prediction}")


Input:    FVNQHLCGSHLVEALYLVCGERGFFYTPKA
Expected: CCCCCCCCHHHHHHHHHHHHHHCECCCCCC
Predicted:CCEEECCCCCCCHHHEEHHCCCCEEECCCC
